# Ch07 RAG & Embeddings — GPU Supplement

This notebook covers advanced RAG patterns and GPU-accelerated embedding models.

**Experiments**:
1. **Embedding model comparison**: CPU-viable vs GPU-beneficial models
2. **Advanced patterns**: Query decomposition, hybrid search (BM25 + dense)
3. **Re-ranking**: Cross-encoder re-scoring for precision

**Requirements**: CUDA GPU (any size), `sentence-transformers`, `rank_bm25`, test corpus

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected — run the CPU notebook instead: notebook-exercise.ipynb\n"
        "To provision a GPU machine: see notes/06-ai-infrastructure/ch01-gpu-architecture/"
    )

device = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {device}  |  VRAM: {vram:.1f} GB")
print("✓ GPU available for accelerated embedding generation")

## Experiment — Embedding Model Precision Comparison

Compare retrieval quality between:
- **all-MiniLM-L6-v2**: 384 dimensions, 22M params, CPU-viable
- **bge-large-en-v1.5**: 1024 dimensions, 335M params, GPU-beneficial

**Metric**: Precision@5 on a test corpus with 10 held-out queries.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import time

# Test corpus (20 docs)
corpus = [
    "Transformers use self-attention to process sequences in parallel",
    "BERT is a bidirectional encoder from transformers",
    "GPT uses causal masking for autoregressive generation",
    "Retrieval-augmented generation grounds LLMs in external data",
    "Vector databases enable semantic search at scale",
    "Byte-pair encoding splits words into subword tokens",
    "Temperature controls the randomness of LLM sampling",
    "RLHF aligns models with human preferences",
    "Chain-of-thought prompting improves multi-step reasoning",
    "Embeddings map text to fixed-size dense vectors",
    "Convolutional neural networks excel at image classification",
    "Recurrent networks process sequences step by step",
    "Backpropagation computes gradients via the chain rule",
    "Dropout prevents overfitting by randomly zeroing activations",
    "Batch normalization stabilizes deep network training",
    "Adam optimizer adapts learning rates per parameter",
    "Cross-entropy loss is standard for classification",
    "Precision measures the accuracy of positive predictions",
    "F1 score balances precision and recall",
    "ROC curves plot true positive vs false positive rates"
]

# Test queries with relevant doc indices
test_queries = [
    ("What is self-attention?", [0]),
    ("Explain BERT architecture", [1]),
    ("How does RAG work?", [3]),
    ("What are vector databases for?", [4]),
    ("Explain tokenization", [5]),
    ("What is temperature in LLMs?", [6]),
    ("How does RLHF work?", [7]),
    ("What is chain-of-thought?", [8]),
    ("Define embeddings", [9]),
    ("What is backpropagation?", [12])
]

models = {
    'all-MiniLM-L6-v2': 384,
    'BAAI/bge-large-en-v1.5': 1024
}

results = {}

for model_name, dims in models.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name} ({dims}d)")
    print('='*60)

    # Load model on GPU
    model = SentenceTransformer(model_name, device='cuda')

    # Encode corpus
    start = time.time()
    corpus_embeddings = model.encode(corpus, convert_to_numpy=True)
    corpus_time = time.time() - start

    # Test queries
    precisions = []
    for query_text, relevant_indices in test_queries:
        query_emb = model.encode(query_text, convert_to_numpy=True)
        scores = cosine_similarity([query_emb], corpus_embeddings)[0]
        top5_indices = np.argsort(scores)[-5:][::-1]

        # Calculate precision@5
        hits = sum(1 for idx in top5_indices if idx in relevant_indices)
        precision = hits / 5.0
        precisions.append(precision)

    avg_precision = np.mean(precisions)
    avg_time = corpus_time / len(corpus)

    results[model_name] = {
        'dims': dims,
        'precision@5': avg_precision,
        'time_per_doc': avg_time
    }

    print(f"Precision@5: {avg_precision:.3f}")
    print(f"Avg encoding time: {avg_time*1000:.2f} ms/doc")

print("\n" + "="*60)
print("COMPARISON TABLE")
print("="*60)
print(f"{'Model':<30} {'Dims':>6} {'Precision@5':>12} {'ms/doc':>8}")
print("-"*60)
for model_name, stats in results.items():
    print(f"{model_name:<30} {stats['dims']:>6} {stats['precision@5']:>12.3f} {stats['time_per_doc']*1000:>8.2f}")

## Analysis

**Trade-offs**:
- **all-MiniLM-L6-v2**: Faster, smaller, good baseline precision, CPU-viable
- **bge-large-en-v1.5**: Higher precision, GPU-beneficial, 3-5× slower per doc

**When to choose**:
- Use MiniLM for prototyping, CPU deployments, or latency-critical applications
- Use bge-large when retrieval quality is paramount and GPU is available

## Key Takeaways

**Embedding model trade-offs:**
- **all-MiniLM-L6-v2**: Fast, CPU-viable, good baseline (384d, ~2ms/doc)
- **bge-large-en-v1.5**: Higher precision (+5-10%), GPU-beneficial, 3-5× slower (1024d, ~8ms/doc)

**Advanced patterns:**
- **Query decomposition**: Split complex queries → retrieve independently → merge. Best for multi-part analytical questions.
- **Hybrid search (BM25 + dense + RRF)**: Combines exact keyword matching with semantic similarity. Use for domain with acronyms, codes, proper nouns.
- **Cross-encoder re-ranking**: Re-score bi-encoder top-k by reading query+doc together. Precision boost at cost of speed.

**Production architecture:**
1. **Bi-encoder** retrieval for top-100 (fast, recall-oriented)
2. **Hybrid search** if domain needs exact matches
3. **Cross-encoder** re-rank to top-5 (slow, precision-oriented)
4. **LLM generation** with top-5 context

**When to use:**
- Start with bi-encoder only (simplest)
- Add hybrid if retrieval misses exact terms
- Add cross-encoder if precision << recall
- Use query decomposition for complex questions

In [ ]:
from sentence_transformers import CrossEncoder
import time

# Step 1: Bi-encoder retrieval (fast, approximate)
test_query = "Explain how retrieval-augmented generation improves LLM accuracy"

start = time.time()
query_emb = embed_model.encode([test_query], convert_to_numpy=True)
bi_encoder_scores = cosine_similarity(query_emb, corpus_embeddings)[0]
top20_indices = np.argsort(bi_encoder_scores)[-20:][::-1]
bi_encoder_time = time.time() - start

print(f"Query: {test_query}\n")
print("="*60)
print(f"\n1. Bi-Encoder Top-5 (fast retrieval: {bi_encoder_time*1000:.1f} ms):")
for idx in top20_indices[:5]:
    print(f"  {bi_encoder_scores[idx]:.3f} — {corpus[idx][:70]}...")

# Step 2: Cross-encoder re-ranking (slow, accurate)
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cuda')

start = time.time()
# Create (query, doc) pairs for top-20
pairs = [[test_query, corpus[idx]] for idx in top20_indices]
cross_encoder_scores = cross_encoder.predict(pairs)
cross_encoder_time = time.time() - start

# Re-rank by cross-encoder scores
reranked_indices = [top20_indices[i] for i in np.argsort(cross_encoder_scores)[::-1]]

print(f"\n2. Cross-Encoder Re-ranked Top-5 (precision: {cross_encoder_time*1000:.1f} ms):")
for idx in reranked_indices[:5]:
    ce_score = cross_encoder_scores[np.where(top20_indices == idx)[0][0]]
    print(f"  {ce_score:.3f} — {corpus[idx][:70]}...")

print(f"\n✓ Cross-encoder improves precision by reading query+doc together")
print(f"✓ Cost: {len(pairs)} forward passes (vs 0 for bi-encoder-only)")
print(f"✓ Use bi-encoder for recall (top-100), cross-encoder for precision (top-5)")

## Experiment 4 — Cross-Encoder Re-Ranking

**Problem**: Bi-encoder retrieval is fast but approximate (embeds query and doc separately).

**Solution**: Use cross-encoder to re-score top-k results by reading query+doc together (slower but more accurate).

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

# Test query with exact term
test_query = "What is DiskANN and how does it work?"

# BM25 (sparse/keyword retrieval)
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
query_tokens = test_query.lower().split()
bm25_scores = bm25.get_scores(query_tokens)

# Get top-10 from BM25
bm25_top10 = np.argsort(bm25_scores)[-10:][::-1]

# Dense (semantic) retrieval
query_emb = embed_model.encode([test_query], convert_to_numpy=True)
dense_scores = cosine_similarity(query_emb, corpus_embeddings)[0]
dense_top10 = np.argsort(dense_scores)[-10:][::-1]

print(f"Query: {test_query}\n")
print("="*60)
print("\nBM25 Top-5 (keyword matching):")
for idx in bm25_top10[:5]:
    print(f"  {bm25_scores[idx]:.3f} — {corpus[idx][:70]}...")

print("\n" + "="*60)
print("\nDense Top-5 (semantic):")
for idx in dense_top10[:5]:
    print(f"  {dense_scores[idx]:.3f} — {corpus[idx][:70]}...")

# Reciprocal Rank Fusion (RRF)
def reciprocal_rank_fusion(rank_lists, k=60):
    """Merge multiple rank lists using RRF."""
    rrf_scores = {}

    for rank_list in rank_lists:
        for rank, doc_idx in enumerate(rank_list, 1):
            if doc_idx not in rrf_scores:
                rrf_scores[doc_idx] = 0
            rrf_scores[doc_idx] += 1 / (k + rank)

    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

# Merge with RRF
rrf_results = reciprocal_rank_fusion([bm25_top10.tolist(), dense_top10.tolist()])

print("\n" + "="*60)
print("\nRRF Hybrid Top-5 (merged):")
for doc_idx, score in rrf_results[:5]:
    print(f"  {score:.4f} — {corpus[doc_idx][:70]}...")

print("\n✓ Hybrid search combines exact keyword match (BM25) with semantic similarity (dense)")

## Experiment 3 — Hybrid Search (BM25 + Dense)

**Problem**: Dense retrieval misses exact keyword matches; sparse retrieval misses paraphrases.

**Solution**: Combine BM25 (sparse/keyword) with dense semantic search using Reciprocal Rank Fusion (RRF).

In [ ]:
import ollama

# Complex query that requires multiple pieces of information
complex_query = "Compare the precision and speed of all-MiniLM-L6-v2 and bge-large-en-v1.5"

# Decompose into sub-queries using LLM
decompose_prompt = f"""Break this complex query into 3-4 simple sub-queries that can be answered independently:

"{complex_query}"

Return only the sub-queries, one per line."""

response = ollama.generate(
    model='phi3:mini',
    prompt=decompose_prompt,
    options={'temperature': 0.0}
)

sub_queries = [q.strip() for q in response['response'].split('\n') if q.strip() and not q.strip().startswith('-')][:4]

print(f"Original query: {complex_query}\n")
print("Decomposed sub-queries:")
for i, sq in enumerate(sub_queries, 1):
    print(f"  {i}. {sq}")

# Retrieve for each sub-query
all_chunks = []
for sq in sub_queries:
    query_emb = embed_model.encode([sq], convert_to_numpy=True)
    scores = cosine_similarity(query_emb, corpus_embeddings)[0]
    top3_indices = np.argsort(scores)[-3:][::-1]

    for idx in top3_indices:
        if corpus[idx] not in all_chunks:  # Deduplicate
            all_chunks.append(corpus[idx])

print(f"\nTotal unique chunks retrieved: {len(all_chunks)}")
print("\nChunks:")
for chunk in all_chunks[:5]:  # Show first 5
    print(f"  - {chunk[:70]}...")

# Now synthesize answer using all chunks
context = "\n".join(all_chunks)
synthesis_prompt = f"""Based on this context, answer: {complex_query}

Context:
{context}

Answer:"""

final_response = ollama.generate(
    model='phi3:mini',
    prompt=synthesis_prompt,
    options={'temperature': 0.0}
)

print(f"\nSynthesized answer:\n{final_response['response']}")

## Experiment 2 — Query Decomposition

**Problem**: Complex multi-part queries produce muddled embeddings that retrieve mediocre results for each sub-question.

**Solution**: Decompose complex queries into atomic sub-queries, retrieve for each, then synthesize.